# Silver Layer Transformation
**Layer:** Silver (Cleaned & Enriched)
**Mục tiêu:**
- Đọc stream từ bảng Bronze.
- Cast kiểu dữ liệu (Date, Number).
- Join bảng Products với bảng Translation để lấy tên tiếng Anh.
- Ghi xuống Silver Delta Tables.

## 1. Khai báo môi trường và tạo Schema Silver.

In [0]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp, trim, upper, initcap

# --- CẤU HÌNH ---
catalog_name = "brazilian_ecommerce"
source_schema = "bronze"
target_schema = "silver"

# Đường dẫn Checkpoint riêng cho Silver (QUAN TRỌNG: Khác Bronze)
checkpoint_base = "/Volumes/brazilian_ecommerce/silver/check_points/"

# Tạo Schema Silver nếu chưa có
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{target_schema}")

print(f"✅ Source: {catalog_name}.{source_schema}")
print(f"✅ Target: {catalog_name}.{target_schema}")
print(f"✅ Checkpoint: {checkpoint_base}")

## 2. Helper Function
Hàm hỗ trợ ghi dữ liệu Streaming xuống bảng Delta Silver.

In [0]:
def write_silver_stream(df, table_name, mode="append"):
    """
    Ghi DataFrame (Streaming) xuống bảng Silver Delta Table.
    """
    checkpoint_loc = f"{checkpoint_base}/{table_name}"
    
    query = (df.writeStream
             .format("delta")
             .outputMode(mode)
             .option("checkpointLocation", checkpoint_loc)
             .option("mergeSchema", "true")
             .trigger(availableNow=True) # Chạy hết data rồi dừng (tiết kiệm chi phí)
             .table(f"{catalog_name}.{target_schema}.{table_name}"))
    
    query.awaitTermination()
    print(f"✅ Finished updating Silver table: {table_name}")

## 3. Transformation: Orders Table
Chuyển đổi các cột `timestamp` từ dạng String sang Timestamp chuẩn để phục vụ Time Intelligence (Daily/Monthly reports).

In [0]:
# 1. Đọc Stream từ Bronze
df_orders_raw = spark.readStream.table(f"{catalog_name}.{source_schema}.orders")

# 2. Transform: Cast String to Timestamp
df_orders_clean = df_orders_raw \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at"))) \
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"))) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"))) \
    .drop("_rescued_data") # Bỏ cột rác của Auto Loader

# 3. Write to Silver
write_silver_stream(df_orders_clean, "orders")

## 4. Transformation: Order Items
Cast `price` và `freight_value` sang kiểu số (Double) để tính toán doanh thu.

In [0]:
df_items_raw = spark.readStream.table(f"{catalog_name}.{source_schema}.order_items")

df_items_clean = df_items_raw \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date"))) \
    .drop("_rescued_data")

write_silver_stream(df_items_clean, "order_items")

## 5. Transformation: Products (Enrichment)
- Cast các cột kích thước/cân nặng sang Integer.
- **Join** với bảng `product_category_name_translation` để lấy tên tiếng Anh (`category_name_english`).

In [0]:
# Đọc bảng Translation dưới dạng STATIC (vì bảng này nhỏ và ít thay đổi)
# Để Join được với Streaming, một bên nên là Static
df_translation = spark.read.table(f"{catalog_name}.{source_schema}.product_category_name_translation")

# Đọc Stream bảng Products
df_products_raw = spark.readStream.table(f"{catalog_name}.{source_schema}.products")

# Thực hiện Join
# Lưu ý: Left Join để giữ lại các sản phẩm không có bản dịch (nếu có)
df_products_enriched = df_products_raw.join(
    df_translation, 
    on="product_category_name", 
    how="left"
)

# Làm sạch và chọn cột cần thiết
df_products_final = df_products_enriched \
    .withColumn("product_name_lenght", col("product_name_lenght").cast("int")) \
    .withColumn("product_description_lenght", col("product_description_lenght").cast("int")) \
    .withColumn("product_photos_qty", col("product_photos_qty").cast("int")) \
    .withColumn("product_weight_g", col("product_weight_g").cast("int")) \
    .withColumn("product_length_cm", col("product_length_cm").cast("int")) \
    .withColumn("product_height_cm", col("product_height_cm").cast("int")) \
    .withColumn("product_width_cm", col("product_width_cm").cast("int")) \
    .select(
        "product_id",
        "product_category_name",
        col("product_category_name_english").alias("category_english"), # Đổi tên cho gọn
        "product_weight_g",
        "product_photos_qty"
    )

write_silver_stream(df_products_final, "products")

## 6. Bulk Transformation for Remaining Tables
Xử lý nhanh các bảng còn lại: `customers`, `sellers`, `order_payments`, `order_reviews`.
Chủ yếu là chuẩn hóa chuỗi (Customers/Sellers) và Cast kiểu số (Payments).

In [0]:
from pyspark.sql.functions import expr

# 1. Customers
df_cust = spark.readStream.table(f"{catalog_name}.{source_schema}.customers") \
    .drop("_rescued_data")
write_silver_stream(df_cust, "customers")

# 2. Sellers
df_sellers = spark.readStream.table(f"{catalog_name}.{source_schema}.sellers") \
    .drop("_rescued_data")
write_silver_stream(df_sellers, "sellers")

# 3. Order Payments
df_pay = spark.readStream.table(f"{catalog_name}.{source_schema}.order_payments") \
    .withColumn("payment_sequential", col("payment_sequential").cast("int")) \
    .withColumn("payment_installments", col("payment_installments").cast("int")) \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .drop("_rescued_data")
write_silver_stream(df_pay, "order_payments")

# 4. Order Reviews (SỬA LỖI TẠI ĐÂY)
# Sử dụng 'try_cast' để biến dữ liệu lỗi thành NULL thay vì báo lỗi
df_rev = spark.readStream.table(f"{catalog_name}.{source_schema}.order_reviews") \
    .withColumn("review_creation_date", expr("try_cast(review_creation_date as timestamp)")) \
    .withColumn("review_answer_timestamp", expr("try_cast(review_answer_timestamp as timestamp)")) \
    .drop("_rescued_data")

# Tùy chọn: Bạn có thể lọc bỏ các dòng bị lỗi Date nếu muốn sạch hoàn toàn
# df_rev = df_rev.filter("review_creation_date IS NOT NULL")

write_silver_stream(df_rev, "order_reviews")

## 7.Xử lý Bảng Geolocation (Aggregation & Deduplication)
## Logic xử lý:

  Read Stream: Đọc dữ liệu từ Bronze.

  Aggregation: Dùng groupBy theo zip_code_prefix.

  Calculations:

  
    Tính trung bình vĩ độ (lat) và kinh độ (lng) để ra tâm của khu vực đó.
    Lấy tên thành phố/bang đầu tiên xuất hiện (first) để làm nhãn hiển thị.

In [0]:
from pyspark.sql.functions import avg, first, col

# --- CẤU HÌNH ---
# (Đảm bảo biến catalog_name, schema_name... đã chạy ở các cell trên)
# Checkpoint riêng cho bảng geolocation vì logic khác biệt (Complete mode)
checkpoint_geo = f"{checkpoint_base}/geolocation"

def process_geolocation_silver():
    print(f"Processing Geolocation -> Deduplicating by Zip Code...")
    
    # 1. Đọc Stream từ Bronze
    df_geo_raw = spark.readStream.table(f"{catalog_name}.{source_schema}.geolocation")
    
    # 2. Thực hiện Aggregation (Gom nhóm)
    # Mục tiêu: Mỗi zip_code chỉ xuất hiện 1 lần với tọa độ trung bình
    df_geo_agg = df_geo_raw \
        .groupBy("geolocation_zip_code_prefix") \
        .agg(
            avg("geolocation_lat").alias("lat"),
            avg("geolocation_lng").alias("lng"),
            first("geolocation_city").alias("city"),
            first("geolocation_state").alias("state")
        )
    
    # 3. Ghi xuống Silver
    # Lưu ý: outputMode("complete") là bắt buộc khi dùng Aggregation mà không có Watermark
    # Nó sẽ ghi đè toàn bộ bảng đích bằng kết quả tính toán mới nhất
    query = (df_geo_agg.writeStream
             .format("delta")
             .outputMode("complete") 
             .option("checkpointLocation", checkpoint_geo)
             .option("mergeSchema", "true")
             .trigger(availableNow=True) # Logic tối ưu chi phí
             .table(f"{catalog_name}.{target_schema}.geolocation"))
    
    query.awaitTermination()
    print(f"✅ Finished Geolocation Deduplication. Table: {catalog_name}.{target_schema}.geolocation")

# Chạy hàm
process_geolocation_silver()

# Kiểm tra kết quả: Đếm số dòng (sẽ ít hơn nhiều so với Bronze)
count_bronze = spark.read.table(f"{catalog_name}.{source_schema}.geolocation").count()
count_silver = spark.read.table(f"{catalog_name}.{target_schema}.geolocation").count()
print(f"📉 Data Reduction: {count_bronze} rows (Bronze) -> {count_silver} unique zip codes (Silver)")

## 8. Cell code tổng hợp này để nhìn lại toàn bộ thành quả tầng Silver

In [0]:
# Cell: Validate toàn bộ tầng Silver
table_list = [
    "customers", "geolocation", "order_items", "order_payments", 
    "order_reviews", "orders", "products", "sellers"
]

print(f"📊 --- REPORT: SILVER LAYER STATUS ({catalog_name}.{target_schema}) ---")
print(f"{'Table Name':<20} | {'Row Count':<15} | {'Status'}")
print("-" * 50)

for tbl in table_list:
    try:
        count = spark.read.table(f"{catalog_name}.{target_schema}.{tbl}").count()
        print(f"{tbl:<20} | {count:<15} | ✅ Ready")
    except Exception as e:
        print(f"{tbl:<20} | {'ERROR':<15} | ❌ {str(e)[0:50]}...")